# Factorised SAC on Three Continuous-Control Environments

This self-contained experiment adapts the repository's factorised value function to continuous actions. A shared critic computes `Q(s, a | task) = phi(s, a)^T psi(task)` and a task-conditioned SAC actor is trained sequentially on three installed dm_control tasks: `cartpole_swingup`, `cheetah_run`, and `walker_walk`. State and action vectors are zero-padded only at the shared-network boundary; the adapter always exposes each environment's native action dimensions.


### Mathematical Factorisation

In continuous control, the critic estimates the action-value function factorised as:
$$ Q(s, a \mid z) = \phi(s, a)^\top \psi(z) $$

where:
* $s$ is the continuous state vector.
* $a$ is the continuous action vector.
* $z$ is the task indicator.
* $\phi(s, a)$ is the state-action embedding, generated by passing the concatenated state and action through an MLP.
* $\psi(z)$ is the task embedding.

The actor in Soft Actor-Critic (SAC) is also task-conditioned, predicting the parameters of a Gaussian policy:
$$ a \sim \pi(\cdot \mid s, z) $$

While the actor shares task-conditioning, the critic applies the dot-product factorisation explicitly, ensuring that value estimates scale structurally across the different dm_control tasks.


In [ ]:
import sys
from pathlib import Path

# Walk up to find the `src/` directory and add it to sys.path
_nb_dir = Path('.').resolve()
for _p in [_nb_dir] + list(_nb_dir.parents):
    if (_p / 'src').is_dir():
        _src = str(_p / 'src')
        if _src not in sys.path:
            sys.path.insert(0, _src)
        break

from utils import set_seed, evaluate_policy
from visualisations import print_goal_embedding_similarity

import random
from collections import deque

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from dm_control import suite

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TASKS = (("cartpole", "swingup"), ("cheetah", "run"), ("walker", "walk"))
TASK_NAMES = tuple(f"{domain}_{task}" for domain, task in TASKS)





class DMControlAdapter(gym.Env):
    """Minimal Gymnasium adapter that exposes flat float32 state observations."""

    def __init__(self, domain, task, seed=0):
        self.domain, self.task, self.seed_value = domain, task, seed
        self._make(seed)

    def _make(self, seed):
        self.env = suite.load(
            self.domain,
            self.task,
            task_kwargs={"random": seed},
            environment_kwargs={"flat_observation": True},
        )
        ts = self.env.reset()
        action = self.env.action_spec()
        self.action_space = gym.spaces.Box(
            action.minimum.astype(np.float32),
            action.maximum.astype(np.float32),
            dtype=np.float32,
        )
        observation = np.asarray(ts.observation, dtype=np.float32).reshape(-1)
        self.observation_space = gym.spaces.Box(
            -np.inf, np.inf, observation.shape, dtype=np.float32
        )

    def reset(self, *, seed=None, options=None):
        if seed is not None:
            self._make(seed)
        ts = self.env.reset()
        return np.asarray(ts.observation, dtype=np.float32).reshape(-1), {}

    def step(self, action):
        ts = self.env.step(np.asarray(action, dtype=np.float32))
        return (
            np.asarray(ts.observation, dtype=np.float32).reshape(-1),
            float(ts.reward or 0.0),
            bool(ts.last()),
            False,
            {},
        )

    def close(self):
        if hasattr(self.env, "close"):
            self.env.close()


def dimensions():
    envs = [DMControlAdapter(*spec) for spec in TASKS]
    answer = (
        max(env.observation_space.shape[0] for env in envs),
        max(env.action_space.shape[0] for env in envs),
    )
    for env in envs:
        env.close()
    return answer


OBS_DIM, ACT_DIM = dimensions()
ACTION_MASKS = torch.zeros(len(TASKS), ACT_DIM, device=DEVICE)
for i, spec in enumerate(TASKS):
    env = DMControlAdapter(*spec)
    ACTION_MASKS[i, : env.action_space.shape[0]] = 1
    env.close()


def task_vector(index):
    value = np.zeros(len(TASKS), dtype=np.float32)
    value[index] = 1
    return value


def pad(vector, size):
    result = np.zeros(size, dtype=np.float32)
    result[: len(vector)] = vector
    return result


class FactorisedCritic(nn.Module):
    """Continuous counterpart of FactorisedDQN_QNetwork."""

    def __init__(self, rep_dim=128):
        super().__init__()
        self.sa_encoder = nn.Sequential(
            nn.Linear(OBS_DIM + ACT_DIM, 256), nn.ReLU(), nn.Linear(256, rep_dim)
        )
        self.goal_encoder = nn.Sequential(
            nn.Linear(len(TASKS), 128), nn.ReLU(), nn.Linear(128, rep_dim)
        )

    def encode_state_action(self, obs, action):
        return F.normalize(
            torch.tanh(self.sa_encoder(torch.cat((obs, action), -1))), dim=-1
        )

    def encode_task(self, task):
        return F.normalize(torch.tanh(self.goal_encoder(task)), dim=-1)

    def forward(self, obs, action, task):
        return (self.encode_state_action(obs, action) * self.encode_task(task)).sum(
            -1, keepdim=True
        )


class TaskConditionedActor(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(OBS_DIM + len(TASKS), 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
        )
        self.mean, self.log_std = nn.Linear(256, ACT_DIM), nn.Linear(256, ACT_DIM)

    def sample(self, obs, task, deterministic=False):
        hidden = self.net(torch.cat((obs, task), -1))
        mean = self.mean(hidden)
        log_std = self.log_std(hidden).clamp(-5, 2)
        distribution = torch.distributions.Normal(mean, log_std.exp())
        latent = mean if deterministic else distribution.rsample()
        squashed = torch.tanh(latent)
        action = squashed
        mask = ACTION_MASKS[task.argmax(-1)]
        action = action * mask
        log_prob = (
            distribution.log_prob(latent) - torch.log(1 - squashed.pow(2) + 1e-6)
        ) * mask
        return action, log_prob.sum(-1, keepdim=True)


class Replay:
    def __init__(self, capacity):
        self.data = deque(maxlen=capacity)

    def add(self, row):
        self.data.append(row)

    def sample(self, batch):
        rows = [
            self.data[i] for i in np.random.choice(len(self.data), batch, replace=False)
        ]
        return tuple(
            torch.as_tensor(np.stack(values), dtype=torch.float32, device=DEVICE)
            for values in zip(*rows)
        )

    def __len__(self):
        return len(self.data)


def evaluate(actor, task_index, episodes=3):
    env = DMControlAdapter(*TASKS[task_index], seed=50_000 + task_index)
    task = torch.as_tensor(task_vector(task_index), device=DEVICE).unsqueeze(0)

    def policy_fn(obs):
        state = torch.as_tensor(pad(obs, OBS_DIM), device=DEVICE).unsqueeze(0)
        with torch.no_grad():
            action, _ = actor.sample(state, task, deterministic=True)
        return action[0, : env.action_space.shape[0]].cpu().numpy()

    mean_return, _ = evaluate_policy(env, policy_fn, episodes=episodes)
    env.close()
    return mean_return


def train_task(
    actor,
    critics,
    targets,
    task_index,
    *,
    total_steps=50_000,
    warmup_steps=2_000,
    batch_size=256,
    seed=42,
):
    """Sequential SAC training for one dm_control task with shared factorised critics."""
    set_seed(seed + task_index)
    env = DMControlAdapter(*TASKS[task_index], seed=seed + task_index)
    q1, q2 = critics
    target1, target2 = targets
    replay = Replay(100_000)
    actor_opt = torch.optim.Adam(actor.parameters(), 3e-4)
    critic_opt = torch.optim.Adam(list(q1.parameters()) + list(q2.parameters()), 3e-4)
    log_alpha = torch.tensor(np.log(0.1), device=DEVICE, requires_grad=True)
    alpha_opt = torch.optim.Adam([log_alpha], 3e-4)
    task_np = task_vector(task_index)
    task = torch.as_tensor(task_np, device=DEVICE).unsqueeze(0)
    history = []
    obs, _ = env.reset()
    for step in range(1, total_steps + 1):
        if step < warmup_steps:
            action = env.action_space.sample()
            padded_action = pad(action, ACT_DIM)
        else:
            state = torch.as_tensor(pad(obs, OBS_DIM), device=DEVICE).unsqueeze(0)
            with torch.no_grad():
                padded_action, _ = actor.sample(state, task)
            padded_action = padded_action[0].cpu().numpy()
            action = padded_action[: env.action_space.shape[0]]
        next_obs, reward, terminated, truncated, _ = env.step(action)
        replay.add(
            (
                pad(obs, OBS_DIM),
                padded_action,
                np.array([reward], np.float32),
                pad(next_obs, OBS_DIM),
                np.array([terminated], np.float32),
                task_np,
            )
        )
        obs = next_obs if not (terminated or truncated) else env.reset()[0]
        if len(replay) >= max(warmup_steps, batch_size):
            states, actions, rewards, next_states, terminateds, tasks = replay.sample(
                batch_size
            )
            with torch.no_grad():
                next_actions, next_logp = actor.sample(next_states, tasks)
                target_value = (
                    torch.min(
                        target1(next_states, next_actions, tasks),
                        target2(next_states, next_actions, tasks),
                    )
                    - log_alpha.exp().detach() * next_logp
                )
                bellman = rewards + 0.99 * (1 - terminateds) * target_value
            q_loss = F.mse_loss(q1(states, actions, tasks), bellman) + F.mse_loss(
                q2(states, actions, tasks), bellman
            )
            critic_opt.zero_grad()
            q_loss.backward()
            nn.utils.clip_grad_norm_(list(q1.parameters()) + list(q2.parameters()), 10)
            critic_opt.step()
            new_actions, logp = actor.sample(states, tasks)
            value = torch.min(
                q1(states, new_actions, tasks), q2(states, new_actions, tasks)
            )
            actor_loss = (log_alpha.exp().detach() * logp - value).mean()
            actor_opt.zero_grad()
            actor_loss.backward()
            actor_opt.step()
            target_entropy = -ACTION_MASKS[task_index].sum()
            alpha_loss = -(log_alpha * (logp.detach() + target_entropy)).mean()
            alpha_opt.zero_grad()
            alpha_loss.backward()
            alpha_opt.step()
            with torch.no_grad():
                for online, target in zip(
                    list(q1.parameters()) + list(q2.parameters()),
                    list(target1.parameters()) + list(target2.parameters()),
                ):
                    target.lerp_(online, 0.005)
        if step % 5_000 == 0:
            score = evaluate(actor, task_index)
            history.append((step, score))
            print(f"{TASK_NAMES[task_index]:18s} step={step:6d} return={score:7.2f}")
    env.close()
    return history


# Small default run; set TOTAL_STEPS=300_000 for the full experiments used in the existing SAC notebooks.
TOTAL_STEPS = 10_000
actor = TaskConditionedActor().to(DEVICE)
critics = (FactorisedCritic().to(DEVICE), FactorisedCritic().to(DEVICE))
targets = (FactorisedCritic().to(DEVICE), FactorisedCritic().to(DEVICE))
for online, target in zip(critics, targets):
    target.load_state_dict(online.state_dict())
results = {
    name: train_task(actor, critics, targets, index, total_steps=TOTAL_STEPS)
    for index, name in enumerate(TASK_NAMES)
}
with torch.no_grad():
    task_embeddings = (
        critics[0].encode_task(torch.eye(len(TASKS), device=DEVICE)).cpu().numpy()
    )
overall_results = {
    name: {"eval_returns": points, "task_embeddings": [task_embeddings[index]]}
    for index, (name, points) in enumerate(results.items())
}
fig, ax = plt.subplots(figsize=(8, 4))
for name, points in results.items():
    if points:
        ax.plot(*zip(*points), marker="o", label=name)
ax.set(
    xlabel="Environment steps within task",
    ylabel="Mean return",
    title="Sequential factorised SAC",
)
ax.grid(alpha=0.25)
ax.legend()
plt.show()
print_goal_embedding_similarity(task_embeddings, goal_labels=TASK_NAMES)
